# Some strategies for extracting features
## Strings

In [ ]:
# Use string length as measure of complexity
# S4E9 Car Price Regression https://www.kaggle.com/competitions/playground-series-s4e9

def get_complexity(df):
    df['model_complexity'] = df['model'].str.len()
    df['engine_complexity'] = df['engine'].str.len()
    df['ext_col_complexity'] = df['ext_col'].str.len()
    df['int_col_complexity'] = df['int_col'].str.len()
    df['transmission_complexity'] = df['transmission'].str.len()
    return df

XY= get_complexity(XY)

In [ ]:
# Use individual string characters as a category
# Russian Car Plates https://www.kaggle.com/competitions/russian-car-plates-prices-prediction

def get_individual_char(df, feature):
    enc = skl.preprocessing.OrdinalEncoder()
    for i in range(6):
        X = df[feature].str[i].values.reshape(-1,1)
        df[f'plate_{i}'] = enc.fit_transform(X)   
        df[f'plate_{i}'] = df[f'plate_{i}'].astype('category')
    return df

XY = get_individual_char(XY,  'numbers')

In [ ]:
# Use regex for pattern recognition
# S4E9 Car Price Regression https://www.kaggle.com/competitions/playground-series-s4e9

def get_feature_patterns(df):
    ### using regex patterns
    pattern = r'(150|250|350|450|550|650)'
    df['model_pickup'] = df['model'].str.extract(pattern).fillna(0).astype(int).astype('category')

    pattern = r'(mpfi|gtdi|tfsi|gdi|pdi|ddi|sidi|di)'
    df['fuel_inject'] = df['engine'].str.extract(pattern).fillna('asp').astype('category')

    pattern = r'(\d+(?:\.\d+)?)hp'
    df['horsepower'] = df['engine'].str.extract(pattern).fillna(-1).astype(float)

    pattern = r'(\d+(?:\.\d+)?)(?=l|\s*lit)'
    df['disp'] = df['engine'].str.extract(pattern).fillna(-1).astype(float)

    pattern = r'([vihsfw]\d+|\d+\s*cyl|straight\s*\d+|flat\s*\d+)'
    df['engine_type'] = df['engine'].str.extract(pattern).fillna('unk').astype('category')
    pattern = r'(\d+)'
    df['cylinders'] = df['engine_type'].str.extract(pattern).fillna(-1).astype(int)

    pattern = r'(\d+(?:\.\d+)?)(?=v)'
    df['valves'] = df['engine'].str.extract(pattern).fillna(-1).astype(int).astype('category')
    df['valves'].replace({697: -1}, inplace=True)

    pattern = r'(turbo|super|cool)'
    df['turbo'] = df['engine'].str.extract(pattern).notna()

    pattern = r'(dohc|sohc|ohv)'
    df['cams'] = df['engine'].str.extract(pattern).fillna('-1').astype('category')

    pattern = r'(\d+)'
    df['gears'] = df['transmission'].str.extract(pattern).fillna(-1).astype(int)

    pattern = r'(cvt|man|auto)'
    df['trans'] = df['transmission'].str.extract(pattern).fillna('-1').astype('category')

    pattern = r'(metallic|coat|cool)'
    df['special_paint'] = df['ext_col'].str.extract(pattern).notna()
    
    return df

XY= get_feature_patterns(XY)

In [ ]:
# Slice string to select informative substrings
# Russian Car Plates https://www.kaggle.com/competitions/russian-car-plates-prices-prediction

def get_plate_codes(df, feature = 'plate', prefix=6):
    df['numbers'] = df[feature].str[:prefix].astype('category')
    df['letters'] = df['numbers'].str[:1] + df['numbers'].str[4:]
    df['letters'] = df['letters'].astype('category')
    df['digits'] = df['numbers'].str[1:4].astype(int)
    
    df['regioncode'] = df[feature].str[prefix:].astype('category')
    regions = {}
    for key, values in REGION_CODES.items():
        for value in values:
            regions[value] = key
    df['region'] = df['regioncode'].replace(to_replace = regions)
    
    print(f"Added new categories based on {feature}")
    return df
    
XY = get_plate_codes(XY)

In [ ]:
# Use string as keys to get values from a dictionary
# Russian Car Plates https://www.kaggle.com/competitions/russian-car-plates-prices-prediction

def get_special(df, GOVERNMENT_CODES=GOVERNMENT_CODES):
    # generate key column in df
    df['special_key'] = df['letters'].astype(str) + "_0_999_" + df['regioncode'].astype(str)

    print(f"{len(GOVERNMENT_CODES.items())} special governement code categories")

    #initialize dictionaries for generating new features
    clean_keys = []
    govcode = {}
    isgovcode = {}
    forbidden = {}
    advantage = {}
    significant = {}
    for key, value in GOVERNMENT_CODES.items():
        # clean key in dictionary to match df column formats
        key_clean = str(key)
        key_clean = key_clean.strip().replace(" ", "").replace("(", "").replace(")", "").replace("'", "").replace(",", "_")
        clean_keys.append(key_clean)

        # update df keys to match key_clean when appropriate
        let = key_clean.split("_")[0]
        low = int(key_clean.split("_")[1])
        hih = int(key_clean.split("_")[2])
        rcd = key_clean.split("_")[3]
        df.loc[(df['letters'] == let) & (df['regioncode'] == rcd) & (df['digits'] >= low) & (df['digits'] <= hih),
               'special_key'] = key_clean

        # update new feature dictionaries
        govcode.update({key_clean: value[0]})
        isgovcode.update({key_clean: True})
        forbidden.update({key_clean: value[1]})
        advantage.update({key_clean: value[2]})
        significant.update({key_clean: value[3]})

    # use dictionaries to generate new features
    df['is_special'] = df['special_key'].map(isgovcode).fillna(False)
    print(f"{len(df[df['is_special']==True])} special plates identified!")
    df['special_govcode'] = df['special_key'].map(govcode).fillna("none")
    df['special_govcode'] = df['special_govcode'].astype('category')
    df['special_forbidden'] = df['special_key'].map(forbidden).fillna(0)
    df['special_advantage'] = df['special_key'].map(advantage).fillna(0)
    df['special_significant'] = df['special_key'].map(significant).fillna(-1)
    return df

XY = get_special(XY)

## Domain Expertise

In [ ]:
# Use models (equations) from domain directly
# S5E5 Calorie Expenditure: https://www.kaggle.com/competitions/playground-series-s5e5

def get_domain_expert_features(df):
    ###https://www.calculator.net/ideal-weight-calculator.html

    df['bmi'] = df['weight']/ ((df['height']/100) **2) 

    # Hanwi Model:
        # Male:   48.0 kg + 2.7 kg per inch over 5 feet
        # Female: 45.5 kg + 2.2 kg per inch over 5 feet
    df['sex'] = df['sex'].replace({'Male':1, 'Female':0})
    df['hanwi'] = df['weight'] - (45.5 + (df['sex'] * (48-45.5)) + (2.2 + (df['sex'] * (2.7-2.2)))*(df['height']-152.4)/2.54)

    # Miller Model:
        # Male:	  56.2 kg + 1.41 kg per inch over 5 feet
        # Female: 53.1 kg + 1.36 kg per inch over 5 feet
    df['miller'] = df['weight'] - (53.1 + (df['sex'] * (56.2-53.1)) + (1.36 + (df['sex'] * (1.41-1.36)))*(df['height']-152.4))

    feats = ['bmi', 'hanwi', 'miller']
    df = mkf.get_transformed_features(df, feats, skl.preprocessing.StandardScaler(), winsorize=[0.001, 0.001])
    return df

XY = get_domain_expert_features(XY)

In [ ]:
# Incorporate new data - add external data set and merge on a feature column
# Russian Car Plates https://www.kaggle.com/competitions/russian-car-plates-prices-prediction

def get_inflation(df, path):
    # create/identify join feature in df
    df['month_year'] = df['date_month'].astype(str) + df['date_year'].astype(str) 
    df['month_year'] = df['month_year'].astype(int)

    # load new data
    df_rubles = pd.read_csv(path)

    # create/identify join feature in new data
    df_rubles['month_year'] = df_rubles['month'].astype(str) + df_rubles['year'].astype(str) 
    df_rubles['month_year'] = df_rubles['month_year'].astype(int)
    df_rubles.drop(columns=['month', 'year'], inplace=True)

    # merge data on join feature
    df = df.merge(df_rubles, how="left", on="month_year")
    df['month_year'] = df['month_year'].astype('category')
    df['inv_rubles'] = 1 / df['rubles']
    return df

inflation_csv = '/kaggle/input/russia-inflation-2021-2025/russia_inflation.csv'

XY = get_inflation(XY, inflation_csv)


In [ ]:
# add categories to show category interactions
# S5E6: Optimal Fertilizer https://www.kaggle.com/competitions/playground-series-s5e6

def get_domain_expert_features(df):
    ### Generate New features based on domain knowledge
    ### https://ipcm.wisc.edu/wp-content/uploads/sites/54/2022/11/SoilTestPK_FINAL.pdf

    # combine categoricals to generate new categoricals
    df['crop_soil_pair'] = df['crop_type'].astype(str) + df['soil_type'].astype(str)

    enc = skl.preprocessing.OrdinalEncoder()
    X = df['crop_soil_pair'].values.reshape(-1,1)
    df['crop_soil_pair'] = enc.fit_transform(X)   
    df['crop_soil_pair'] = df['crop_soil_pair'].astype(int).astype('category')

    return df

XY = get_domain_expert_features(XY)

In [ ]:
# Use models (equations) from domain directly
# use binary counter to create new categories
# S5E8: Bank Marketing https://www.kaggle.com/competitions/playground-series-s5e8


def get_expert_features(df):
    ###expected balance = nominal_balance * (0.06 * age - 1) for age < 50 and 2 for age > 50
    expected_balance = df['balance'].median()
    df['age_balance'] = df['balance'] -  ((df['age'].clip(upper=50) * 0.06 - 1) * expected_balance)

    ### proxy credit score
    df['debt_score'] = df['housing'].eq(True).astype(int)
    df['debt_score'] += df['balance'].gt(0).astype(int) * 2
    df['debt_score'] += df['loan'].eq(False).astype(int) * 4
    df['debt_score'] += df['default'].eq(False).astype(int) * 8

    ### Simplfy highly skewed features

    df['prior_contact'] = df['pdays'].ne(-1)
    df['previous_contact'] = df['previous'].ne(0)

    df['pweek'] = df['pdays'] // 7
    df['pmonth'] = df['pdays'] // 31 
    df['pmonth'] = df['pmonth'].clip(upper=24) 

    df['campaign_clip'] = df['campaign'].clip(upper=48)

    df['age_clip'] = (df['age'] + 2) //5
    df['age_clip'] = df['age_clip'].clip(upper=15)

    return df

XY = get_expert_features(XY)




In [ ]:
# slice numeric features and use binary counter to create new categories
# S5E6: Optimal Fertilizer https://www.kaggle.com/competitions/playground-series-s5e6

def get_domain_expert_features(df):
    #### https://www.gardenmyths.com/fertilizer-selecting-the-right-npk-ratio/
    ### Fertilizer N-P-K -> is soil level above or below mean?
    
    # use numerical features to build categoricals
    df['soil_needs'] = (
        (df['potassium'] > 9).astype(int) * 1
        + (df['phosphorous'] > 21).astype(int) * 2
        + (df['nitrogen'] > 23).astype(int) * 4
    )
    df['soil_needs'] = df['soil_needs'].astype('category')

    # use numerical features to build categoricals
    df['soil_has'] = (
        (df['potassium'] < 6).astype(int) * 1
        + (df['phosphorous'] < 11).astype(int) * 2
        + (df['nitrogen'] < 16).astype(int) * 4
        + (df['potassium'] > 12).astype(int) * 8
        + (df['phosphorous'] > 30).astype(int) * 16
        + (df['nitrogen'] > 30).astype(int) * 32
    )
    df['soil_has'] = df['soil_has'].astype('category')

    return df

XY = get_domain_expert_features(XY)

## Use PCA inspired features

In [ ]:
# PCA loading analysis
# S5E5 Calorie Expenditure https://www.kaggle.com/competitions/playground-series-s5e5

def get_pca_inspired_features(df):
    ## delta from reference
    df['body_temp'] +=  -37
    df['heart_rate'] +=  -60
    ### effort 
    df['effort'] = df['heart_rate'] * df['body_temp'] * df['duration']

    ### effort per size -> PCA 1
    df['size_effort'] = df['effort'] / (df['weight'] * df['height'])
    ### total effort -> PCA 2
    df['max_effort'] = df['effort'] * df['weight'] * df['height']
    ### temp vs heart rate -> PCA 5
    df['vital_ratio'] = df['heart_rate'] / df['body_temp']
    ### workout effort per unit time -> PCA 6
    df['burn_rate'] = df['body_temp'] * df['heart_rate'] / df['duration'] 
    ### healthy height weight ratio -> PCA 7
    df['height_weight_ratio'] = df['weight']/df['height']

    feats = ['height_weight_ratio', 'effort', 'size_effort', 'max_effort', 'vital_ratio', 'burn_rate']
    df = mkf.get_transformed_features(df, feats , skl.preprocessing.PowerTransformer(), winsorize=[0.001, 0.001])

    return df

XY = get_pca_inspired_features(XY)

#### Creating Features Lessons

In [ ]:
def get_features(df):
    #mathmatical relationships
    #Sums and Ratios
    df["LivLotRatio"] = df['GrLivArea']/df['LotArea']
    df["Spaciousness"] = (df['FirstFlrSF'] + df['SecondFlrSF'])/df['TotRmsAbvGrd']
    df["TotalOutsideSF"] = df['WoodDeckSF'] + df['OpenPorchSF'] + df['EnclosedPorch'] + df['Threeseasonporch'] + df['ScreenPorch']

    #categorical interactions -> OHE and then multiply by numeric feature
    df_dummies = pd.get_dummies(df.BldgType, prefix="Bldg")
    df_dummies = df_dummies.mul(df.GrLivArea, axis=0)
    df =df.join(df_dummies)

    #count feature
    df["PorchTypes"] = df[['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'Threeseasonporch', 'ScreenPorch']].gt(0).sum(axis = 1)

    #grab element of string
    df['MSClass'] = df.MSSubClass.str.split('_', n = 1, expand = True)[0]

    #groupby
    df["MedNhbdArea"] = df.groupby('Neighborhood')['GrLivArea'].transform('median')

    return df




In [ ]:
#https://contrib.scikit-learn.org/category_encoders/mestimate.html

from category_encoders import MEstimateEncoder

def get_target_encode(df, features, target):
    X_encode = df[features + [target]].sample(frac=0.20, random_state=0)
    y_encode = X_encode.pop(target)
    
    X_pretrain = df.drop(X_encode.index)
    y_train = X_pretrain.pop(target)
    
    encoder = MEstimateEncoder(cols = ['MSSubClass'], m = 3)

    # Fit the encoder on the encoding split
    encoder.fit(X_encode, y_encode)

    X_train = encoder.transform(X_pretrain, y_train)



In [ ]:
### lag features for time series data
### need to clean and validate

### get two days of history and the difference (change per day) in the feature
def get_history(df, feature, delta_only = True):
    #yesterday's feature
    df[f'{feature}_yesterday'] = df[feature].shift(periods = 1)
    df[f'{feature}_yesterday'].fillna(method='bfill', inplace = True)
    #feature 2 days prior
    df[f'{feature}_tdby'] = df[feature].shift(periods = 2)
    df[f'{feature}_tdby'].fillna(method='bfill', inplace = True)
    #change from yesterday and change from 2 days ago
    df[f'{feature}_yesterday_delta'] = df[feature] - df[f'{feature}_yesterday']
    df[f'{feature}_tdby_delta'] = 0.5 * (df[feature] - df[f'{feature}_tdby'])
    if delta_only == True:
        df.drop(f'{feature}_yesterday', inplace = True, axis = 1)
        df.drop(f'{feature}_tdby', inplace = True, axis = 1)
    return df

In [ ]:
df = average_sales.to_frame()

time = np.arange(len(df.index))  # time dummy

df['time'] = time

X = df.loc[:, ['time']]  # features
y = df.loc[:, 'sales']  # target

model = LinearRegression()
model.fit(X, y)

y_pred = pd.Series(model.predict(X), index=X.index)